# Build double ellipse fano forward model

In [ ]:
from __future__ import annotations
import json
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
PACKAGE = Path.cwd().parent
WORK = PACKAGE / 'work'
PROJECT = PACKAGE
FIT_ZERO = PACKAGE / 'inputs/fit_zero'
FIT_COATED = PACKAGE / 'inputs/fit_coated'

FIT_OUT = WORK / 'combined_fano_fits'
MODEL_OUT = WORK / 'fano_forward_model_double_ellipse_S843_theta205_0to6nm_parameter_noise0p05'
PARAMETERS = ['fanoCentreNm', 'fanoGammaNm', 'fanoQ', 'fanoOffset', 'fanoAmplitude']
NOISY_PARAMETERS = ['fanoCentreNm', 'fanoGammaNm', 'fanoQ', 'fanoAmplitude']


In [ ]:
zero = pd.read_csv(FIT_ZERO / 'fano_thickness_fit_results.csv')
coated = pd.read_csv(FIT_COATED / 'fano_thickness_fit_results.csv')
fits = pd.concat([zero, coated], ignore_index=True).sort_values('thicknessNm')
fits['spectrumIndex'] = np.arange(1, len(fits) + 1)

FIT_OUT.mkdir(parents=True, exist_ok=True)
MODEL_OUT.mkdir(parents=True, exist_ok=True)

fits.to_csv(FIT_OUT / 'fano_thickness_fit_results.csv', index=False)

thickness_data = fits.thicknessNm.to_numpy(dtype=float)

coefficients = np.polynomial.polynomial.polyfit(thickness_data, fits[PARAMETERS].to_numpy(dtype=float), 2)
coefficient_table = pd.DataFrame({'parameterName': PARAMETERS, 'quadraticA': coefficients[2], 'linearB': coefficients[1], 'constantC': coefficients[0]})

coefficient_table.to_csv(MODEL_OUT / 'fano_forward_model_coefficient_functions.csv', index=False)
thickness = np.linspace(0.0, 6.0, 61)
values = np.polynomial.polynomial.polyval(thickness, coefficients).T
clean = pd.DataFrame(values, columns=PARAMETERS)

clean.insert(0, 'thicknessNm', thickness)
clean.to_csv(MODEL_OUT / 'fano_forward_model_clean_parameters.csv', index=False)

noise_std = {name: 0.05 * float(np.ptp(clean[name])) if name in NOISY_PARAMETERS else 0.0 for name in PARAMETERS}

pd.DataFrame({'parameterName': PARAMETERS, 
              'noiseStd': [noise_std[name] for name in PARAMETERS], 
              'fanoParameterNoiseSeed': 7}).to_csv(MODEL_OUT / 'fano_forward_model_parameter_noise_config.csv', index=False)

wavelength = np.linspace(680.0, 750.0, 7001)
centre, gamma, q, offset, amplitude = values.T
epsilon = (wavelength[:, None] - centre) / gamma
spectra = offset - amplitude * (q + epsilon) ** 2 / ((1.0 + q * q) * (1.0 + epsilon * epsilon))

spectrum_table = pd.DataFrame(spectra, columns=[f't_{value:.2f}nm'.replace('.', 'p') for value in thickness])
spectrum_table.insert(0, 'wavelengthNm', wavelength)
spectrum_table.to_csv(MODEL_OUT / 'fano_forward_model_clean_spectrum_matrix.csv', index=False)

fit_quality = {}


In [ ]:
for index, name in enumerate(PARAMETERS):
    predicted = np.polynomial.polynomial.polyval(thickness_data, coefficients[:, index])
    observed = fits[name].to_numpy(dtype=float)
    residual = np.sum((observed - predicted) ** 2)
    total = np.sum((observed - observed.mean()) ** 2)
    fit_quality[name] = float(1.0 - residual / total)

manifest = {'model': 'DoubleEllipis S843 theta205, thickness 0-6 nm', 'source_files': [str(PROJECT / 'inputs' / 'raw' / 'DoubleEllipis_S843_theta205.txt'), str(PROJECT / 'inputs' / 'raw' / 'DoubleEllipis_S843_theta205_thickness1-6nm.txt')], 'source_thickness_nm': thickness_data.tolist(), 'parameter_fit': 'quadratic least squares over the seven fitted spectra', 'parameter_fit_r_squared': fit_quality, 'parameter_noise': "independent zero-mean Gaussian; sigma is 5% of each fitted parameter's 0-6 nm peak-to-peak range", 'parameter_noise_std': noise_std, 'offset_noise_std': 0.0, 'wavelength_range_nm': [680.0, 750.0], 'thickness_range_nm': [0.0, 6.0]}

(MODEL_OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))

print(json.dumps(manifest, indent=2))
